# NB Assist — Task 2, run on a Kaggle GPU

This notebook produces the **model-stack** numbers for the submission: the local multilingual retriever (`istiaqfuad/triBne-e5-small`, MIT) plus the local generator (`Qwen/Qwen3-1.7B`, Apache-2.0, 4-bit) answering the supplied `bank-faq.csv`, then the full 40-question evaluation.

**Read this before judging anything below**

- Nothing here reaches a customer. The questions come from the pack's `test-questions.csv`; the project contains no customer or company data.
- Model **weights** are downloaded from Hugging Face on first run, so internet must be on. Inference itself is local on the T4 — the offline system never calls an inference API.
- The 40 questions are **held out**. They are used only for measurement, never for training, tuning, or prompt selection.
- The embedder is a third-party fine-tune of `intfloat/multilingual-e5-small`, trained on CC-BY parallel pairs, so outside language data already reaches retrieval. Disclosed in `README.md`.
- Section 7 calls an **online** AI model. That is the brief's own comparison requirement, and it sends only those 40 synthetic sample questions.
- Section 6 is the brief's *"Extra (optional): train the small model further"*. Sections 1–5 and 7 are the mandatory part.
- The retrieval fixes for **Banglish** are in `src/`, not in this notebook. The Kaggle dataset is a copy of the project, so re-upload it after changing `src/`: section 3 asserts the staged copy is the current one, because the last stale copy reported 97.5% overall while answering none of the Banglish questions.
- Section 5 reports the 40 questions **per script** as well as overall. A whole writing system can disappear into an aggregate, and it did.


## 1. Install dependencies

In [ ]:
# Find the uploaded dataset regardless of Kaggle's generated mount prefix.
from pathlib import Path
_candidates = list(Path('/kaggle/input').glob('**/requirements-kaggle.txt'))
assert _candidates, 'requirements-kaggle.txt was not found in the attached dataset'
!pip -q install -r {_candidates[0]}


## 2. Copy the project to a writable directory

`/kaggle/input` is read-only, so `evaluate.py` cannot write its report there. The next cell stages the project into `/kaggle/working`. It accepts either the original `src/`, `data/`, and `ui/` folders or a flat Kaggle upload where those files are all directly under `biai-bank-assistant`.


In [ ]:
import os
import shutil
from pathlib import Path

SRC = Path(os.environ.get('BIAI_KAGGLE_DATASET', str(next(Path('/kaggle/input').glob('**/requirements-kaggle.txt')).parent)))
DST = Path('/kaggle/working/nb_assist')

# /kaggle/input is read-only, so stage a writable project copy.
if DST.exists():
    shutil.rmtree(DST)
DST.mkdir(parents=True)


def copy_item(relative_name):
    source = SRC / relative_name
    target = DST / relative_name
    if source.is_dir():
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(source, target, ignore=shutil.ignore_patterns('__pycache__'))
    elif source.exists():
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, target)


# Preferred layout: the dataset preserves src/, data/, and ui/ directories.
if (SRC / 'src' / 'assistant.py').exists():
    for item in ('src', 'ui', 'data', 'training_data', 'evaluate.py', 'train.py', 'kaggle_demo.py',
                 'api_server.py', 'README.md', 'ONE_PAGE_NOTE.md'):
        copy_item(item)
else:
    # Compatibility layout: Kaggle flattened the uploaded files into one folder.
    # Rebuild the package directories from their unique filenames.  We create
    # fresh __init__.py files because the original src/__init__.py and
    # ui/__init__.py cannot safely coexist after flattening.
    src_dir = DST / 'src'
    data_dir = DST / 'data'
    ui_dir = DST / 'ui'
    src_dir.mkdir()
    data_dir.mkdir()
    ui_dir.mkdir()

    for name in ('assistant.py', 'data.py', 'generator.py', 'models.py',
                 'normalize.py', 'retrieval.py', 'remote.py', 'bn_render.py'):
        source = SRC / name
        assert source.exists(), f'flat upload is missing {name}'
        shutil.copy2(source, src_dir / name)
    (src_dir / '__init__.py').write_text(
        '"""Offline bilingual bank assistant package."""\n'
        '\ntry:\n'
        '    from . import bn_render as _bn_render\n'
        '    _bn_render.install()\n'
        'except Exception as _exc:\n'
        '    print("[bn_render] not installed:", type(_exc).__name__, _exc)\n',
        encoding='utf-8',
    )

    # The two BIAI files are uploaded exactly as the pack delivered them. There is
    # deliberately no third data file: bank-faq.csv is the only source the
    # assistant may answer from, so the Bangla and Banglish reply has to be
    # written by the model below, not read off the disk.
    for name in ('bank-faq.csv', 'test-questions.csv'):
        source = SRC / name
        assert source.exists(), f'flat upload is missing {name}'
        shutil.copy2(source, data_dir / name)

    for name in ('main_window.py', 'theme.py'):
        source = SRC / name
        if source.exists():
            shutil.copy2(source, ui_dir / name)
    (ui_dir / '__init__.py').write_text('', encoding='utf-8')

    # Preserve the local Vashantor language-only corpus when Kaggle flattens
    # the upload; it is training text, never an answer source.
    if (SRC / 'vashantor_train.jsonl').exists():
        (DST / 'training_data').mkdir(exist_ok=True)
        shutil.copy2(SRC / 'vashantor_train.jsonl', DST / 'training_data' / 'vashantor_train.jsonl')
        if (SRC / 'vashantor_train.metadata.json').exists():
            shutil.copy2(SRC / 'vashantor_train.metadata.json', DST / 'training_data' / 'vashantor_train.metadata.json')

    for name in ('evaluate.py', 'kaggle_demo.py', 'api_server.py', 'train.py', 'README.md', 'ONE_PAGE_NOTE.md'):
        source = SRC / name
        if source.exists():
            shutil.copy2(source, DST / name)

# The install cell needs this file at the dataset root.
assert (SRC / 'requirements-kaggle.txt').exists(), \
    'requirements-kaggle.txt must be uploaded at the dataset root'
shutil.copy2(SRC / 'requirements-kaggle.txt', DST / 'requirements-kaggle.txt')

# Always stage the current defensive trainer from this notebook. This prevents an
# old Kaggle dataset upload from reintroducing the gated-dataset crash.
TRAIN_SOURCE = "\"\"\"Train the optional local-language adapter used by NB Assist.\n\nThe script deliberately has a narrow data contract: the supplied FAQ is used as\nthe factual domain corpus, while public licensed language corpora supply\nBangla/Banglish language supervision.  The held-out ``test-questions.csv`` is\nnever opened.  The resulting LoRA adapter is loaded by ``LocalGenerator`` when\n``BIAI_ADAPTER_PATH`` points at its directory.\n\nThis is intended for a Kaggle T4 (the same command is used by the notebook):\n\n    python train.py --faq data/bank-faq.csv --output /kaggle/working/nb-adapter\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nimport re\nimport shutil\nfrom pathlib import Path\n\n\nBENGALI_RE = re.compile(r\"[\\u0980-\\u09FF]\")\nBANGlish_MARKERS = frozenset(\n    \"amar apnar apnader koto kibhabe kivabe korte korbo korini janabo lage lagbe dorkar hobe hoy kobe ache ase pabo paben pete dite pari parben kon koy kothay jonno theke er din taka tulate harie bhule nomini joggo joggota shomoy shakha gulo tulate sud\".split()\n)\n\n\ndef _args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\"--faq\", type=Path, required=True)\n    parser.add_argument(\"--output\", type=Path, required=True)\n    parser.add_argument(\"--model\", default=\"Qwen/Qwen3-1.7B\")\n    parser.add_argument(\"--max-pairs\", type=int, default=1200)\n    parser.add_argument(\"--max-banking-intents\", type=int, default=1200)\n    parser.add_argument(\"--max-banglabridge\", type=int, default=200)\n    parser.add_argument(\n        \"--vashantor\",\n        type=Path,\n        default=Path(__file__).resolve().parent / \"training_data\" / \"vashantor_train.jsonl\",\n    )\n    parser.add_argument(\"--epochs\", type=float, default=2.0)\n    return parser.parse_args()\n\n\ndef _parallel_pairs(max_pairs: int) -> list[tuple[str, str, str]]:\n    \"\"\"Return (source, target, direction) examples from the public corpus.\"\"\"\n    from datasets import load_dataset\n\n    try:\n        kwargs = {\"token\": os.environ[\"HF_TOKEN\"]} if os.environ.get(\"HF_TOKEN\") else {}\n        stream = load_dataset(\n            \"istiaqfuad/bangla-english-banglish-pairs\", split=\"train\", streaming=True, **kwargs\n        )\n    except Exception as exc:\n        print(f\"warning: skipping public parallel corpus ({type(exc).__name__}: {exc})\")\n        return []\n    rows: list[tuple[str, str, str]] = []\n    seen: set[tuple[str, str, str]] = set()\n    for row in stream:\n        anchor, positive = (row.get(\"anchor\") or \"\").strip(), (row.get(\"positive\") or \"\").strip()\n        if not anchor or not positive:\n            continue\n        a_bn, p_bn = bool(BENGALI_RE.search(anchor)), bool(BENGALI_RE.search(positive))\n        if a_bn == p_bn:\n            continue\n        bangla, latin = (anchor, positive) if a_bn else (positive, anchor)\n        if not (6 <= len(bangla) <= 240 and 8 <= len(latin) <= 240):\n            continue\n        # Keep the filter domain-oriented without using the held-out questions.\n        is_domain = bool(re.search(\n            r\"account|bank|loan|card|balance|interest|payment|transfer|deposit|branch|fee|charge|limit|withdraw|credit|debit|statement|cheque|pin|transaction|money|taka\",\n            latin,\n            re.IGNORECASE,\n        ))\n        is_banglish = any(token in BANGlish_MARKERS for token in re.findall(r\"[A-Za-z]+\", latin.lower()))\n        if not is_domain and not is_banglish:\n            continue\n        if not is_banglish:\n            # Generic English banking pairs.\n            directions = ((latin, bangla, \"english_to_bangla\"), (bangla, latin, \"bangla_to_english\"))\n        else:\n            # Romanised Bangla pairs are valuable for the register the base\n            # model misses. They are labelled separately so the adapter does not\n            # learn to call Banglish English.\n            directions = ((bangla, latin, \"bangla_to_banglish\"), (latin, bangla, \"banglish_to_bangla\"))\n        if not directions:\n            continue\n        for source, target, direction in directions:\n            key = (source, target, direction)\n            if key not in seen:\n                seen.add(key)\n                rows.append(key)\n        if len(rows) >= max_pairs * 2:\n            break\n    return rows\n\n\ndef _local_vashantor(path: Path, max_rows: int = 500) -> list[tuple[str, str, str]]:\n    \"\"\"Load only the local archive's Train-derived, language-only pairs.\"\"\"\n    if not path.exists():\n        return []\n    rows: list[tuple[str, str, str]] = []\n    with path.open(encoding=\"utf-8\") as handle:\n        for line in handle:\n            item = json.loads(line)\n            bn, bl, en = item.get(\"bangla\", \"\"), item.get(\"banglish\", \"\"), item.get(\"english\", \"\")\n            if not (bn and bl and en):\n                continue\n            rows.extend(((en, bn, \"english_to_bangla\"), (bn, en, \"bangla_to_english\"),\n                         (bn, bl, \"bangla_to_banglish\"), (bl, bn, \"banglish_to_bangla\")))\n            if len(rows) >= max_rows * 4:\n                break\n    return rows\n\n\ndef _public_style_examples(max_banking: int, max_banglabridge: int) -> list[tuple[str, str, str]]:\n    \"\"\"Use synthetic, licensed language examples without importing their facts.\"\"\"\n    from datasets import load_dataset\n\n    rows: list[tuple[str, str, str]] = []\n    token = os.environ.get(\"HF_TOKEN\")\n    datasets_to_try = [\n        (\"learn-abc/banking14-intents-en-bn-banglish\", max_banking, \"banking_register\"),\n        (\"subhajitmahata84/banglabridge-instructions\", max_banglabridge, \"banglish_instruction\"),\n    ]\n    for dataset_id, limit, direction in datasets_to_try:\n        try:\n            kwargs = {\"token\": token} if token else {}\n            stream = load_dataset(dataset_id, split=\"train\", streaming=True, **kwargs)\n            used = 0\n            for item in stream:\n                if direction == \"banking_register\":\n                    source = target = (item.get(\"text\") or \"\").strip()\n                else:\n                    source = (item.get(\"instruction\") or \"\").strip()\n                    target = (item.get(\"output\") or \"\").strip()\n                if source and target:\n                    rows.append((source, target, direction))\n                    used += 1\n                if used >= limit:\n                    break\n        except Exception as exc:\n            print(f\"warning: skipping {dataset_id} ({type(exc).__name__}: {exc})\")\n    return rows\n\n\ndef _faq_examples(faq_path: Path) -> list[tuple[str, str, str]]:\n    from src.data import load_faq\n\n    examples = []\n    for entry in load_faq(faq_path):\n        examples.append((\n            f\"FAQ answer: {entry.answer}\\nRestate the FAQ answer in plain English.\",\n            entry.answer,\n            \"faq_grounding\",\n        ))\n    return examples\n\n\ndef main() -> int:\n    args = _args()\n    # Imports are delayed so the repository remains runnable in lexical smoke\n    # mode on a CPU-only laptop with only PySide6 installed.\n    import torch\n    from datasets import Dataset\n    from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training\n    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer, TrainingArguments\n\n    pairs = _parallel_pairs(args.max_pairs)\n    local_pairs = _local_vashantor(args.vashantor)\n    public_style = _public_style_examples(args.max_banking_intents, args.max_banglabridge)\n    examples = _faq_examples(args.faq) + pairs + local_pairs + public_style\n    if not (pairs or local_pairs or public_style):\n        raise RuntimeError(\"No language-training examples were available\")\n    # A previous failed run can leave an empty output directory. Remove it so a\n    # later notebook cell never mistakes that directory for a valid adapter.\n    if args.output.exists():\n        shutil.rmtree(args.output)\n    args.output.mkdir(parents=True, exist_ok=True)\n\n    tokenizer = AutoTokenizer.from_pretrained(args.model)\n    if tokenizer.pad_token is None:\n        tokenizer.pad_token = tokenizer.eos_token\n    quant = BitsAndBytesConfig(\n        load_in_4bit=True,\n        bnb_4bit_quant_type=\"nf4\",\n        bnb_4bit_compute_dtype=torch.float16,\n        bnb_4bit_use_double_quant=True,\n    )\n    base = AutoModelForCausalLM.from_pretrained(\n        args.model, device_map=\"auto\", dtype=torch.float16, quantization_config=quant\n    )\n    base.config.use_cache = False\n    model = get_peft_model(\n        prepare_model_for_kbit_training(base),\n        LoraConfig(\n            task_type=TaskType.CAUSAL_LM,\n            r=16,\n            lora_alpha=32,\n            lora_dropout=0.05,\n            target_modules=[\"q_proj\", \"k_proj\", \"v_proj\", \"o_proj\"],\n        ),\n    )\n\n    ignore = -100\n    def encode(batch):\n        result = {\"input_ids\": [], \"attention_mask\": [], \"labels\": []}\n        for source, target, direction in zip(batch[\"source\"], batch[\"target\"], batch[\"direction\"]):\n            prompt = (\n                f\"Task: {direction.replace('_', ' ')}.\\n\"\n                f\"Source:\\n{source}\\n\"\n                \"Target (answer only):\\n\"\n            )\n            p = tokenizer(prompt, add_special_tokens=False)[\"input_ids\"][:384]\n            t = tokenizer(target + tokenizer.eos_token, add_special_tokens=False)[\"input_ids\"][:128]\n            ids = p + t\n            result[\"input_ids\"].append(ids)\n            result[\"attention_mask\"].append([1] * len(ids))\n            result[\"labels\"].append([ignore] * len(p) + t)\n        return result\n\n    class Pad:\n        def __call__(self, features):\n            width = max(len(item[\"input_ids\"]) for item in features)\n            return {\n                \"input_ids\": torch.tensor([x[\"input_ids\"] + [tokenizer.pad_token_id] * (width - len(x[\"input_ids\"])) for x in features]),\n                \"attention_mask\": torch.tensor([x[\"attention_mask\"] + [0] * (width - len(x[\"attention_mask\"])) for x in features]),\n                \"labels\": torch.tensor([x[\"labels\"] + [ignore] * (width - len(x[\"labels\"])) for x in features]),\n            }\n\n    dataset = Dataset.from_dict({\n        \"source\": [row[0] for row in examples],\n        \"target\": [row[1] for row in examples],\n        \"direction\": [row[2] for row in examples],\n    }).map(encode, batched=True, remove_columns=[\"source\", \"target\", \"direction\"])\n    trainer = Trainer(\n        model=model,\n        args=TrainingArguments(\n            output_dir=str(args.output),\n            per_device_train_batch_size=2,\n            gradient_accumulation_steps=8,\n            num_train_epochs=args.epochs,\n            learning_rate=2e-4,\n            fp16=torch.cuda.is_available(),\n            optim=\"paged_adamw_8bit\",\n            save_strategy=\"no\",\n            report_to=\"none\",\n        ),\n        train_dataset=dataset,\n        data_collator=Pad(),\n        processing_class=tokenizer,\n    )\n    trainer.train()\n    model.eval()\n    model.config.use_cache = True\n    model.save_pretrained(args.output)\n    (args.output / \"training_manifest.json\").write_text(\n        json.dumps({\n            \"base_model\": args.model,\n            \"faq\": str(args.faq),\n            \"public_dataset\": \"istiaqfuad/bangla-english-banglish-pairs (CC-BY-4.0)\",\n            \"additional_public_datasets\": [\n                \"learn-abc/banking14-intents-en-bn-banglish (MIT; train split)\",\n                \"subhajitmahata84/banglabridge-instructions (CC-BY-4.0; train split)\",\n                \"Vashantor train split (CC-BY-4.0; local archive)\",\n            ],\n            \"parallel_examples\": len(pairs),\n            \"local_vashantor_examples\": len(local_pairs),\n            \"public_style_examples\": len(public_style),\n            \"faq_examples\": len(_faq_examples(args.faq)),\n            \"held_out_test_questions_used\": False,\n        }, indent=2),\n        encoding=\"utf-8\",\n    )\n    print(f\"saved adapter to {args.output} ({len(examples)} examples)\")\n    return 0\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n"
(DST / 'train.py').write_text(TRAIN_SOURCE, encoding='utf-8')
print('staged train.py:', len(TRAIN_SOURCE), 'bytes')

os.makedirs(DST / 'reports', exist_ok=True)
missing = [
    relative for relative in (
        'src/assistant.py', 'src/generator.py', 'train.py', 'training_data/vashantor_train.jsonl', 'data/bank-faq.csv',
        'data/test-questions.csv', 'evaluate.py',
    ) if not (DST / relative).exists()
]
extra_data = sorted(p.name for p in (DST / 'data').iterdir())
print('input files:', sorted(p.name for p in SRC.iterdir()))
print('staged project:', sorted(p.name for p in DST.iterdir()))
print('data files:', extra_data)
assert not missing, f'upload is missing required files after staging: {missing}'

# The supplied FAQ must arrive as it was supplied, and nothing may sit beside it.
import csv as _csv
with (DST / 'data' / 'bank-faq.csv').open(encoding='utf-8-sig', newline='') as handle:
    supplied_header = next(_csv.reader(handle))
assert supplied_header == ['id', 'topic', 'question', 'answer'], (
    f'data/bank-faq.csv was uploaded with extra columns {supplied_header}. The '
    'supplied file is the only answer source and is not to be edited: a Bangla '
    'reply has to come from the model, not from a file of prepared answers.'
)


## 3. Build the assistant

Both models are loaded explicitly here, and the assertions matter. If either one fails to load, the system silently degrades to lexical smoke-test mode and a Bangla question then gets an English verbatim-FAQ reply — which is exactly the failure that hid the scoring bug in the first draft of this submission.

In [ ]:
import sys

assert (DST / 'src' / 'assistant.py').exists(), (
    'Run the staging cell above first; src/assistant.py was not found in ' + str(DST)
)
sys.path.insert(0, str(DST))
os.environ['BIAI_EMBEDDING_MODEL'] = 'istiaqfuad/triBne-e5-small'
os.environ['BIAI_GENERATION_MODEL'] = 'Qwen/Qwen3-1.7B'
os.environ['BIAI_TRANSLATOR_MODEL'] = 'SifatAlMukit/finetuned-en-bn-nllb'
os.environ['BIAI_RENDER_MAX_NEW_TOKENS'] = '96'

from src.assistant import BankAssistant
from src.data import load_faq

assistant = BankAssistant(load_faq(os.path.join(DST, 'data', 'bank-faq.csv')))

# LocalGenerator loads lazily; explicitly load it before checking availability.
assistant.generator.load()

print('retrieval mode     :', assistant.retriever.mode)
print('generator available:', assistant.generator.available)
print('generator status   :', assistant.generator.load_error or 'loaded')
assert assistant.retriever.dense_available, 'embedding model did not load'
if not assistant.generator.available:
    raise RuntimeError(
        'Qwen generator failed to load: ' + (assistant.generator.load_error or 'no error was recorded')
    )

import torch
print('device             :', assistant.generator.model.device)
print('quantised          :', getattr(assistant.generator.model, 'is_loaded_in_4bit', 'unknown'))
print('parameters         :', sum(p.numel() for p in assistant.generator.model.parameters())/1e9, 'B')

# The dataset is a *copy* of this project, so a stale upload runs old retrieval
# and nothing here would say so. These assertions are what makes that loud.
from src.retrieval import CHANNEL_FLOOR, DENSE_WEIGHT, UNKNOWN_TERM_WEIGHT

assert {"en", "bn", "banglish", "mixed"} <= set(DENSE_WEIGHT), (
    'the staged src/retrieval.py predates the language-aware fusion - re-upload '
    'the dataset with the current src/ folder'
)
assert DENSE_WEIGHT['banglish'] < DENSE_WEIGHT['en'], (
    'Banglish is weighted as though a Bangla/English embedder had been trained on '
    'Roman-script Bangla; it has not'
)
assert UNKNOWN_TERM_WEIGHT > 0, (
    'staged src still treats a query term that no FAQ contains as free, which is '
    'what let a two-word overlap score as a full match'
)

print('\nembedder share of the fused score, by the script the customer used')
for script in ('en', 'bn', 'banglish', 'mixed'):
    print(f'  {script:9s} dense share={assistant.retriever.dense_weight(script):.2f}'
          f'  support floor={assistant.retriever.support_floor(script):.4f}'
          f'  (per-channel floors {CHANNEL_FLOOR})')


# The gates that fixed the Banglish and Bangla replies live in src/, not here, so
# these are the checks that say the upload carries them. A stale upload would
# still print respectable aggregate numbers while reciting the customer's own
# question back at them.
from inspect import signature

from src.normalize import answer_is_wellformed, informative_terms, reuses_faq_wording
from src.retrieval import is_ambiguous

assert 'source' in signature(answer_is_wellformed).parameters, (
    'staged src/normalize.py predates the wording gate: re-upload the dataset '
    'with the current src/ folder'
)
# The eligibility question must reach the FAQ as one concept per idea, not as a
# Bangla word and its English gloss counted twice.
terms = informative_terms('পার্সোনাল লোনের জন্য কী যোগ্যতা লাগে? personal loan eligibility')
assert not ('পার্সোনাল' in terms and 'personal' in terms), (
    'staged src/normalize.py has no concept dedup, which is what let the loan-'
    'interest entry win the loan-eligibility question'
)
# bank-faq.csv is English, so a Bangla reply cannot be word-matched against it.
# An upload that still tried would send every correct Bangla answer back to
# English, and the language column of section 5 would collapse. This reply is in
# the right script, complete, not an echo, and states the entry's only figure.
assert answer_is_wellformed(
    'এসএমএস অ্যালার্টের খরচ বছরে BDT 120।',
    'bn',
    'এসএমএস অ্যালার্টের জন্য কত চার্জ?',
    'SMS alerts cost a flat BDT 120 per year, charged to your account.',
) is True, (
    'staged src/normalize.py still judges Bangla wording against the English FAQ, '
    'which rejects every good Bangla reply and serves English instead'
)
assert is_ambiguous([]) is False, (
    'staged src/retrieval.py still calls every 0.003 gap a coin flip, which hands '
    'off questions the auditable channels actually separate'
)
print('answer gates present: concept dedup, cross-script wording rule, auditable tie-break')


## 4. One question per script

The brief's requirements 1–4 in five calls: answers only from the FAQ, a reply in the language the customer used, everything on the small open model, and a hand-off for anything the FAQ cannot support.

In [ ]:
examples = [
    'Can I open a joint account with my spouse?',
    'ক্রেডিট কার্ডের বিল কবে দিতে হয়?',
    'Card er PIN bhule gechi, reset korbo kibhabe?',
    'Personal loan er interest rate koto?',
    "What is today's dollar to taka exchange rate?",
]

for question in examples:
    result = assistant.answer(question)
    print(f'Q   : {question}')
    print(f'A   : {result.answer}')
    print(f'      status={result.status}  source_id={result.source_id}  language={result.language}')
    print(f'      reason={result.support_type}  mode={result.metadata["mode"]}  '
          f'answer_form={result.metadata.get("answer_form", "-")}  '
          f'latency={result.latency_ms:.0f} ms\n')

### 4.1 The Banglish path, on its own

Banglish is where this pipeline broke on the GPU while the headline number stayed respectable.
Three separate things caused it, and widening the glossary would not have fixed any of them: the
fused score handed 0.62 of the weight to a Bangla/English embedder that has never seen Roman-script Bangla; the support floor
was calibrated on that fused scale, so a Banglish query started life below the bar; and
`detect_language` was fitted to *questions*, so a model-written Banglish *reply* read as English
and the language gate turned a correct answer into a hand-off. The checks below are structural.
The table measures, it does not decide anything.

In [ ]:
import csv

from evaluate import EXPECTED_SOURCE
from src.generator import REFUSALS
from src.normalize import answer_matches_language, detect_language

# A refusal is held to the same language check as a generated answer. The
# Banglish one used to score as English, which would have made a correct
# hand-off look like a language failure.
for script, message in REFUSALS.items():
    assert answer_matches_language(message, script), (
        f'{script} refusal fails its own language check: {message!r}'
    )
print('refusal strings detected as:', {script: detect_language(message) for script, message in REFUSALS.items()})

# Retrieval-only view of every Banglish query in the held-out set. Printed
# separately because the aggregate is exactly where this failure hid.
floor = assistant.retriever.support_floor('banglish')
test_rows = list(csv.DictReader(
    open(os.path.join(DST, 'data', 'test-questions.csv'), encoding='utf-8-sig')))
print(f'\nBanglish queries against the {assistant.retriever.mode} retriever, floor {floor:.3f}:')
for row in test_rows:
    if detect_language(row['question']) != 'banglish':
        continue
    question_id = int(row['id'])
    hit = assistant.retriever.search(row['question'], top_k=1)[0]
    expected = EXPECTED_SOURCE.get(question_id)
    label = f'expected={expected:2d}' if expected else 'expected=hand-off'
    outcome = 'ok' if not expected or hit.entry.faq_id == expected else 'WRONG'
    print(f"  id={question_id:02d} top1={hit.entry.faq_id:2d} {label} "
          f"score={hit.score:.3f} {outcome:5s} {row['question']}")


## 5. The 40-question evaluation (mandatory)

`evaluate.py` runs the whole held-out set and writes `reports/offline_results.json` into the writable copy. Latency here is model-stack latency: retrieval plus two generator calls per question (an entailment check, then the answer).

In [ ]:
os.chdir(DST)
!python evaluate.py

In [ ]:
import json

report = json.load(open(os.path.join(DST, 'reports', 'offline_results.json'), encoding='utf-8'))
rows = report['rows']
answered = [r for r in rows if not r['expected_unknown']]
misses = [r for r in rows if not (r['source_correct'] and r['refusal_correct'])]
forms = {}
for r in answered:
    forms[r['answer_form']] = forms.get(r['answer_form'], 0) + 1
model_written_non_english = sum(
    1 for r in rows if r['status'] == 'answered' and r['language'] in {'bn', 'banglish'} and r['answer_form'] == 'model'
)

print(f"retrieval mode        {report['retrieval_mode']}")
print(f"generator loaded      {report['generator_loaded']}")
print(f"correct FAQ source    {sum(r['source_correct'] for r in answered)}/{len(answered)}")
print(f"correct hand-offs     {sum(r['refusal_correct'] for r in rows if r['expected_unknown'])}/{report['unknown']}")
print(f"overall gate accuracy {report['overall_gate_accuracy']:.1%}")
print(f"language accuracy     {report['language_accuracy']:.1%}")
print(f"replies free of the customer's own question {report['echo_free_accuracy']:.1%}")
print(f"replies that finish their sentence         {report['complete_answer_accuracy']:.1%}")
print(f"median / p95 latency  {report['median_latency_ms']:.0f} ms / {report['p95_latency_ms']:.0f} ms")
print(f"answer forms          {forms}")
print(f"non-English answers written by the model: {model_written_non_english}")

# One line per script. The overall gate accuracy is what let a total Banglish
# failure look like a passing run, so the split is printed before the misses.
print('\nper script:')
for script in sorted({r['language'] for r in rows}):
    group = [r for r in rows if r['language'] == script]
    answerable = [r for r in group if not r['expected_unknown']]
    print(f"  {script:9s} n={len(group):2d}"
          f"  source {sum(r['source_correct'] for r in answerable)}/{len(answerable)}"
          f"  gate {sum(r['source_correct'] and r['refusal_correct'] for r in group)}/{len(group)}"
          f"  reply in the customer's language {sum(r['language_correct'] for r in group)}/{len(group)}"
          f"  answered {sum(r['status'] == 'answered' for r in group):2d}"
          f"  model-written {sum(r['answer_form'] == 'model' for r in group):2d}")

print('\nmisses:')
for r in misses:
    print(f"  id={r['id']:02d} got={r['source_id']} expected={r['expected_source_id']} "
          f"status={r['status']} reason={r['support_type']}  {r['question']}")
reasons = sorted({r['support_type'] for r in rows if r['support_type']})
print('\nhand-off reasons:', {reason: sum(1 for r in rows if r['support_type'] == reason) for reason in reasons})


## 6. Train the small model on allowed language and banking data

This is the optional training step from the brief. It uses the supplied FAQ as a small factual-domain corpus and a disclosed public CC-BY Bangla/English/Banglish corpus for language supervision. It never opens `test-questions.csv`, and it does not use customer or real bank data. The resulting LoRA adapter is loaded by `LocalGenerator` through `BIAI_ADAPTER_PATH`; section 5 is then rerun so the app and the report use the trained stack.


In [ ]:
# Release the optional renderer model before QLoRA training.
try:
    from src import bn_render as _render_layer
    _render_layer.RENDERER.unload()
except Exception as exc:
    print('translator was not loaded, nothing to release:', type(exc).__name__)
import gc
import torch
gc.collect()
torch.cuda.empty_cache()

# This cell writes the current defensive trainer itself. It does not depend on
# which directory the previous notebook cell left as the working directory.
from pathlib import Path as _Path
DST = _Path('/kaggle/working/nb_assist')
DST.mkdir(parents=True, exist_ok=True)
TRAIN_SOURCE = "\"\"\"Train the optional local-language adapter used by NB Assist.\n\nThe script deliberately has a narrow data contract: the supplied FAQ is used as\nthe factual domain corpus, while public licensed language corpora supply\nBangla/Banglish language supervision.  The held-out ``test-questions.csv`` is\nnever opened.  The resulting LoRA adapter is loaded by ``LocalGenerator`` when\n``BIAI_ADAPTER_PATH`` points at its directory.\n\nThis is intended for a Kaggle T4 (the same command is used by the notebook):\n\n    python train.py --faq data/bank-faq.csv --output /kaggle/working/nb-adapter\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nimport re\nimport shutil\nfrom pathlib import Path\n\n\nBENGALI_RE = re.compile(r\"[\\u0980-\\u09FF]\")\nBANGlish_MARKERS = frozenset(\n    \"amar apnar apnader koto kibhabe kivabe korte korbo korini janabo lage lagbe dorkar hobe hoy kobe ache ase pabo paben pete dite pari parben kon koy kothay jonno theke er din taka tulate harie bhule nomini joggo joggota shomoy shakha gulo tulate sud\".split()\n)\n\n\ndef _args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\"--faq\", type=Path, required=True)\n    parser.add_argument(\"--output\", type=Path, required=True)\n    parser.add_argument(\"--model\", default=\"Qwen/Qwen3-1.7B\")\n    parser.add_argument(\"--max-pairs\", type=int, default=1200)\n    parser.add_argument(\"--max-banking-intents\", type=int, default=1200)\n    parser.add_argument(\"--max-banglabridge\", type=int, default=200)\n    parser.add_argument(\n        \"--vashantor\",\n        type=Path,\n        default=Path(__file__).resolve().parent / \"training_data\" / \"vashantor_train.jsonl\",\n    )\n    parser.add_argument(\"--epochs\", type=float, default=2.0)\n    return parser.parse_args()\n\n\ndef _parallel_pairs(max_pairs: int) -> list[tuple[str, str, str]]:\n    \"\"\"Return (source, target, direction) examples from the public corpus.\"\"\"\n    from datasets import load_dataset\n\n    try:\n        kwargs = {\"token\": os.environ[\"HF_TOKEN\"]} if os.environ.get(\"HF_TOKEN\") else {}\n        stream = load_dataset(\n            \"istiaqfuad/bangla-english-banglish-pairs\", split=\"train\", streaming=True, **kwargs\n        )\n    except Exception as exc:\n        print(f\"warning: skipping public parallel corpus ({type(exc).__name__}: {exc})\")\n        return []\n    rows: list[tuple[str, str, str]] = []\n    seen: set[tuple[str, str, str]] = set()\n    for row in stream:\n        anchor, positive = (row.get(\"anchor\") or \"\").strip(), (row.get(\"positive\") or \"\").strip()\n        if not anchor or not positive:\n            continue\n        a_bn, p_bn = bool(BENGALI_RE.search(anchor)), bool(BENGALI_RE.search(positive))\n        if a_bn == p_bn:\n            continue\n        bangla, latin = (anchor, positive) if a_bn else (positive, anchor)\n        if not (6 <= len(bangla) <= 240 and 8 <= len(latin) <= 240):\n            continue\n        # Keep the filter domain-oriented without using the held-out questions.\n        is_domain = bool(re.search(\n            r\"account|bank|loan|card|balance|interest|payment|transfer|deposit|branch|fee|charge|limit|withdraw|credit|debit|statement|cheque|pin|transaction|money|taka\",\n            latin,\n            re.IGNORECASE,\n        ))\n        is_banglish = any(token in BANGlish_MARKERS for token in re.findall(r\"[A-Za-z]+\", latin.lower()))\n        if not is_domain and not is_banglish:\n            continue\n        if not is_banglish:\n            # Generic English banking pairs.\n            directions = ((latin, bangla, \"english_to_bangla\"), (bangla, latin, \"bangla_to_english\"))\n        else:\n            # Romanised Bangla pairs are valuable for the register the base\n            # model misses. They are labelled separately so the adapter does not\n            # learn to call Banglish English.\n            directions = ((bangla, latin, \"bangla_to_banglish\"), (latin, bangla, \"banglish_to_bangla\"))\n        if not directions:\n            continue\n        for source, target, direction in directions:\n            key = (source, target, direction)\n            if key not in seen:\n                seen.add(key)\n                rows.append(key)\n        if len(rows) >= max_pairs * 2:\n            break\n    return rows\n\n\ndef _local_vashantor(path: Path, max_rows: int = 500) -> list[tuple[str, str, str]]:\n    \"\"\"Load only the local archive's Train-derived, language-only pairs.\"\"\"\n    if not path.exists():\n        return []\n    rows: list[tuple[str, str, str]] = []\n    with path.open(encoding=\"utf-8\") as handle:\n        for line in handle:\n            item = json.loads(line)\n            bn, bl, en = item.get(\"bangla\", \"\"), item.get(\"banglish\", \"\"), item.get(\"english\", \"\")\n            if not (bn and bl and en):\n                continue\n            rows.extend(((en, bn, \"english_to_bangla\"), (bn, en, \"bangla_to_english\"),\n                         (bn, bl, \"bangla_to_banglish\"), (bl, bn, \"banglish_to_bangla\")))\n            if len(rows) >= max_rows * 4:\n                break\n    return rows\n\n\ndef _public_style_examples(max_banking: int, max_banglabridge: int) -> list[tuple[str, str, str]]:\n    \"\"\"Use synthetic, licensed language examples without importing their facts.\"\"\"\n    from datasets import load_dataset\n\n    rows: list[tuple[str, str, str]] = []\n    token = os.environ.get(\"HF_TOKEN\")\n    datasets_to_try = [\n        (\"learn-abc/banking14-intents-en-bn-banglish\", max_banking, \"banking_register\"),\n        (\"subhajitmahata84/banglabridge-instructions\", max_banglabridge, \"banglish_instruction\"),\n    ]\n    for dataset_id, limit, direction in datasets_to_try:\n        try:\n            kwargs = {\"token\": token} if token else {}\n            stream = load_dataset(dataset_id, split=\"train\", streaming=True, **kwargs)\n            used = 0\n            for item in stream:\n                if direction == \"banking_register\":\n                    source = target = (item.get(\"text\") or \"\").strip()\n                else:\n                    source = (item.get(\"instruction\") or \"\").strip()\n                    target = (item.get(\"output\") or \"\").strip()\n                if source and target:\n                    rows.append((source, target, direction))\n                    used += 1\n                if used >= limit:\n                    break\n        except Exception as exc:\n            print(f\"warning: skipping {dataset_id} ({type(exc).__name__}: {exc})\")\n    return rows\n\n\ndef _faq_examples(faq_path: Path) -> list[tuple[str, str, str]]:\n    from src.data import load_faq\n\n    examples = []\n    for entry in load_faq(faq_path):\n        examples.append((\n            f\"FAQ answer: {entry.answer}\\nRestate the FAQ answer in plain English.\",\n            entry.answer,\n            \"faq_grounding\",\n        ))\n    return examples\n\n\ndef main() -> int:\n    args = _args()\n    # Imports are delayed so the repository remains runnable in lexical smoke\n    # mode on a CPU-only laptop with only PySide6 installed.\n    import torch\n    from datasets import Dataset\n    from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training\n    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer, TrainingArguments\n\n    pairs = _parallel_pairs(args.max_pairs)\n    local_pairs = _local_vashantor(args.vashantor)\n    public_style = _public_style_examples(args.max_banking_intents, args.max_banglabridge)\n    examples = _faq_examples(args.faq) + pairs + local_pairs + public_style\n    if not (pairs or local_pairs or public_style):\n        raise RuntimeError(\"No language-training examples were available\")\n    # A previous failed run can leave an empty output directory. Remove it so a\n    # later notebook cell never mistakes that directory for a valid adapter.\n    if args.output.exists():\n        shutil.rmtree(args.output)\n    args.output.mkdir(parents=True, exist_ok=True)\n\n    tokenizer = AutoTokenizer.from_pretrained(args.model)\n    if tokenizer.pad_token is None:\n        tokenizer.pad_token = tokenizer.eos_token\n    quant = BitsAndBytesConfig(\n        load_in_4bit=True,\n        bnb_4bit_quant_type=\"nf4\",\n        bnb_4bit_compute_dtype=torch.float16,\n        bnb_4bit_use_double_quant=True,\n    )\n    base = AutoModelForCausalLM.from_pretrained(\n        args.model, device_map=\"auto\", dtype=torch.float16, quantization_config=quant\n    )\n    base.config.use_cache = False\n    model = get_peft_model(\n        prepare_model_for_kbit_training(base),\n        LoraConfig(\n            task_type=TaskType.CAUSAL_LM,\n            r=16,\n            lora_alpha=32,\n            lora_dropout=0.05,\n            target_modules=[\"q_proj\", \"k_proj\", \"v_proj\", \"o_proj\"],\n        ),\n    )\n\n    ignore = -100\n    def encode(batch):\n        result = {\"input_ids\": [], \"attention_mask\": [], \"labels\": []}\n        for source, target, direction in zip(batch[\"source\"], batch[\"target\"], batch[\"direction\"]):\n            prompt = (\n                f\"Task: {direction.replace('_', ' ')}.\\n\"\n                f\"Source:\\n{source}\\n\"\n                \"Target (answer only):\\n\"\n            )\n            p = tokenizer(prompt, add_special_tokens=False)[\"input_ids\"][:384]\n            t = tokenizer(target + tokenizer.eos_token, add_special_tokens=False)[\"input_ids\"][:128]\n            ids = p + t\n            result[\"input_ids\"].append(ids)\n            result[\"attention_mask\"].append([1] * len(ids))\n            result[\"labels\"].append([ignore] * len(p) + t)\n        return result\n\n    class Pad:\n        def __call__(self, features):\n            width = max(len(item[\"input_ids\"]) for item in features)\n            return {\n                \"input_ids\": torch.tensor([x[\"input_ids\"] + [tokenizer.pad_token_id] * (width - len(x[\"input_ids\"])) for x in features]),\n                \"attention_mask\": torch.tensor([x[\"attention_mask\"] + [0] * (width - len(x[\"attention_mask\"])) for x in features]),\n                \"labels\": torch.tensor([x[\"labels\"] + [ignore] * (width - len(x[\"labels\"])) for x in features]),\n            }\n\n    dataset = Dataset.from_dict({\n        \"source\": [row[0] for row in examples],\n        \"target\": [row[1] for row in examples],\n        \"direction\": [row[2] for row in examples],\n    }).map(encode, batched=True, remove_columns=[\"source\", \"target\", \"direction\"])\n    trainer = Trainer(\n        model=model,\n        args=TrainingArguments(\n            output_dir=str(args.output),\n            per_device_train_batch_size=2,\n            gradient_accumulation_steps=8,\n            num_train_epochs=args.epochs,\n            learning_rate=2e-4,\n            fp16=torch.cuda.is_available(),\n            optim=\"paged_adamw_8bit\",\n            save_strategy=\"no\",\n            report_to=\"none\",\n        ),\n        train_dataset=dataset,\n        data_collator=Pad(),\n        processing_class=tokenizer,\n    )\n    trainer.train()\n    model.eval()\n    model.config.use_cache = True\n    model.save_pretrained(args.output)\n    (args.output / \"training_manifest.json\").write_text(\n        json.dumps({\n            \"base_model\": args.model,\n            \"faq\": str(args.faq),\n            \"public_dataset\": \"istiaqfuad/bangla-english-banglish-pairs (CC-BY-4.0)\",\n            \"additional_public_datasets\": [\n                \"learn-abc/banking14-intents-en-bn-banglish (MIT; train split)\",\n                \"subhajitmahata84/banglabridge-instructions (CC-BY-4.0; train split)\",\n                \"Vashantor train split (CC-BY-4.0; local archive)\",\n            ],\n            \"parallel_examples\": len(pairs),\n            \"local_vashantor_examples\": len(local_pairs),\n            \"public_style_examples\": len(public_style),\n            \"faq_examples\": len(_faq_examples(args.faq)),\n            \"held_out_test_questions_used\": False,\n        }, indent=2),\n        encoding=\"utf-8\",\n    )\n    print(f\"saved adapter to {args.output} ({len(examples)} examples)\")\n    return 0\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n"
(DST / 'train.py').write_text(TRAIN_SOURCE, encoding='utf-8')
assert 'skipping public parallel corpus' in (DST / 'train.py').read_text(encoding='utf-8')
print('using defensive trainer:', (DST / 'train.py').stat().st_size, 'bytes')

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
    if token:
        import os
        os.environ['HF_TOKEN'] = token
except Exception:
    pass

import os
os.chdir(DST)
!python train.py --faq data/bank-faq.csv --output /kaggle/working/nb-assist-adapter --max-pairs 1200 --epochs 2


In [ ]:
# Point the same app surface at the trained adapter and rerun the held-out evaluation.
from pathlib import Path
adapter_path = Path('/kaggle/working/nb-assist-adapter')
adapter_config = adapter_path / 'adapter_config.json'
if not adapter_config.exists():
    raise RuntimeError(
        f'No trained adapter found at {adapter_config}. Re-run the training cell above and wait for it to finish.'
    )
os.environ['BIAI_ADAPTER_PATH'] = str(adapter_path)
from src.assistant import BankAssistant
from src.data import load_faq
trained_assistant = BankAssistant(load_faq(os.path.join(DST, 'data', 'bank-faq.csv')))
trained_assistant.generator.load()
assert trained_assistant.generator.available, trained_assistant.generator.load_error
print('trained adapter loaded:', os.environ['BIAI_ADAPTER_PATH'])
# evaluate.py constructs the same backend from these environment variables.


In [ ]:
# Evaluate the actual trained answer path.
os.chdir(DST)
!python evaluate.py


In [ ]:
import json
report = json.load(open(os.path.join(DST, 'reports', 'offline_results.json'), encoding='utf-8'))
print('trained model:', report['generator_loaded'])
print('correct FAQ source:', sum(r['source_correct'] for r in report['rows'] if not r['expected_unknown']), '/ 30')
print('correct hand-offs:', sum(r['refusal_correct'] for r in report['rows'] if r['expected_unknown']), '/ 10')
print('reply language accuracy:', report['language_accuracy'])
print('format accuracy:', report['format_accuracy'])
print('median / p95 latency:', report['median_latency_ms'], '/', report['p95_latency_ms'], 'ms')
print('median / p95 renderer latency:', report.get('median_render_latency_ms', 0), '/', report.get('p95_render_latency_ms', 0), 'ms')
print('per-script:', report['language_accuracy_by_script'])


## 7. Mandatory — the same 40 questions on an online AI model, then compare

> *"Then run the same 40 questions on any online AI model and compare."*

This is the only section that talks to an outside service, and it sends **only** the 40 synthetic questions from the supplied pack. Add a key under **notebook Settings → Add-ons and Secrets → Secrets → New Secret**, named `GEMINI_API_KEY`, value from [Google AI Studio](https://aistudio.google.com/apikey) (free tier is enough for 40 calls). The cell reads it with `UserSecretsClient`, so the key never appears in the notebook text or the git history. Keep internet on. Another hosted model can be used instead by setting `BIAI_BASELINE_MODEL`.

The comparison is kept fair, and where a machine cannot judge it, it is left visibly open:

- The online model gets **no FAQ**. It is asked as a generic assistant and told to answer with exactly `HANDOFF` when it does not know this bank's rule. That is the behaviour the brief wants compared.
- Correct hand-off rate on the 10 uncovered questions and latency are computed automatically.
- Invented figures are computed automatically: any number in an online answer that appears nowhere in the FAQ.
- Whether its **content** is right on the 30 answerable questions needs a human read. All 40 answers are saved to `reports/baseline_results.csv` for that, and the table below says `needs human review` rather than inventing a score.

In [ ]:
BASELINE_MODEL = os.environ.get('BIAI_BASELINE_MODEL', 'gemini-3.5-flash-lite')

# Only this section needs the hosted client, so it is deliberately not in
# requirements-kaggle.txt: the offline run must not depend on it.
!pip -q install google-genai


def api_key():
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret('GEMINI_API_KEY')
        if key:
            return key
    except Exception:
        pass
    return os.environ.get('GEMINI_API_KEY') or None


client, key = None, api_key()
if key:
    try:
        from google import genai
        client = genai.Client(api_key=key)
    except ImportError as exc:
        print('google-genai did not import:', exc)
print('baseline model:', BASELINE_MODEL, '| key found:', bool(key))
if client is None:
    print('no key: section 7 will print the manual fallback instead of failing')


In [ ]:
import csv
import time

from evaluate import UNKNOWN_IDS
from src.data import load_faq
from src.normalize import detect_language
from src.retrieval import extract_numbers

FAQ_NUMBERS = set()
for entry in load_faq(os.path.join(DST, 'data', 'bank-faq.csv')):
    FAQ_NUMBERS |= extract_numbers(entry.searchable_text)

PROMPT = (
    "You are a bank chat assistant. Answer the customer directly in the language they wrote in. "
    "If you do not actually know this bank's specific rule, reply with exactly HANDOFF and nothing "
    "else.\n\nCustomer: {question}\n\nAnswer:"
)
baseline_path = os.path.join(DST, 'reports', 'baseline_results.csv')

if client is None:
    print('Online run skipped. Two ways to still complete this mandatory comparison:')
    print('  1. add the GEMINI_API_KEY secret, enable internet, re-run the two cells above; or')
    print('  2. paste the 40 questions into any hosted chat model, save its 40 replies in the same')
    print(f'     order in a CSV with the columns used below, and put that file at {baseline_path}.')
else:
    rows = []
    for row in csv.DictReader(open(os.path.join(DST, 'data', 'test-questions.csv'), encoding='utf-8-sig')):
        question_id, question = int(row['id']), row['question']
        started = time.perf_counter()
        try:
            answer = (client.models.generate_content(model=BASELINE_MODEL,
                                                     contents=PROMPT.format(question=question)).text or '').strip()
        except Exception as exc:
            answer = f'ERROR {type(exc).__name__}: {exc}'
        elapsed = (time.perf_counter() - started) * 1000
        handed_off = answer.upper().startswith('HANDOFF')
        rows.append({
            'id': question_id,
            'question': question,
            'answer': answer,
            'expected_unknown': question_id in UNKNOWN_IDS,
            'handed_off': handed_off,
            'refusal_correct': handed_off == (question_id in UNKNOWN_IDS),
            'language': detect_language(question),
            'invented_figures': sorted(extract_numbers(answer) - FAQ_NUMBERS) if not handed_off else [],
            'latency_ms': elapsed,
        })
        print(f'{question_id:02d} handoff={handed_off!s:5} {elapsed:7.0f} ms  {answer[:90]!r}')
    with open(baseline_path, 'w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    print('\nsaved', baseline_path)

In [ ]:
import ast
import csv
import json
import os
import statistics

from evaluate import UNKNOWN_IDS
from src.data import load_faq
from src.retrieval import extract_numbers

BASELINE_MODEL = os.environ.get('BIAI_BASELINE_MODEL', 'gemini-3.5-flash-lite')
FAQ_NUMBERS = set()
for entry in load_faq(os.path.join(DST, 'data', 'bank-faq.csv')):
    FAQ_NUMBERS |= extract_numbers(entry.searchable_text)

path = os.path.join(DST, 'reports', 'baseline_results.csv')
rows = list(csv.DictReader(open(path, encoding='utf-8'))) if os.path.exists(path) else []
if not rows:
    print('nothing to compare yet: run the cell above, or place a manual CSV at', path)
else:
    def as_bool(value):
        return str(value).strip().lower() in {'1', 'true', 'yes'}

    answered = [r for r in rows if not as_bool(r['expected_unknown'])]
    uncovered = [r for r in rows if as_bool(r['expected_unknown'])]
    online_refusals = sum(as_bool(r['handed_off']) for r in uncovered)
    def figure_list(value):
        try:
            parsed = ast.literal_eval(value or '[]')
        except (SyntaxError, ValueError):
            parsed = []
        return parsed if isinstance(parsed, list) else []

    online_invented = sum(bool(figure_list(r['invented_figures'])) for r in answered)
    online_latency = [float(r['latency_ms']) for r in rows if str(r.get('latency_ms', '')).replace('.', '', 1).isdigit()]

    offline = json.load(open(os.path.join(DST, 'reports', 'offline_results.json'), encoding='utf-8'))
    offline_rows = offline['rows']
    offline_answered = [r for r in offline_rows if not r['expected_unknown']]
    offline_invented = sum(bool(extract_numbers(r['answer']) - FAQ_NUMBERS) for r in offline_answered)

    print('| | NB Assist — local Qwen3-1.7B + MarianMT Bangla renderer, offline | Online AI model |')
    print('|---|---|---|')
    print(f'| model | {os.environ.get("BIAI_GENERATION_MODEL", "Qwen/Qwen3-1.7B")} (4-bit, on a T4) | {BASELINE_MODEL} (hosted) |')
    print(f'| customer data leaves the machine | no | yes (40 sample questions only) |')
    print(f'| correct hand-offs on the 10 uncovered | '
          f'{sum(r["refusal_correct"] for r in offline_rows if r["expected_unknown"])}/{len([r for r in offline_rows if r["expected_unknown"]])} '
          f'| {online_refusals}/{len(uncovered)} |')
    print(f'| correct FAQ source on the 30 answerable | '
          f'{sum(r["source_correct"] for r in offline_answered)}/{len(offline_answered)} '
          f'| needs human review — answers are in {os.path.basename(path)} |')
    print(f'| answers containing a figure absent from the FAQ | {offline_invented}/{len(offline_answered)} (blocked by the grounding gate) '
          f'| {online_invented}/{len(answered)} |')
    if online_latency:
        print(f'| median latency | {offline["median_latency_ms"]:.0f} ms | {statistics.median(online_latency):.0f} ms (network included) |')
    print()
    print('The defensible claim is that the offline system cannot print an unsourced number, not that')
    print('its prose is better than a hosted model. Read the CSV before writing the sentence.')

## 8. Mandatory — point the PySide6 desktop app at this notebook

The brief's system is offline by design, and a laptop CPU writes Bangla badly, so
this is how the desktop app gets the GPU's answer without shipping a GPU. The app
keeps asking the same `answer(question)` surface; `BIAI_API_URL` decides whether
that call is served by `src/assistant.py` on the laptop or by this notebook.

Run the cell below. It prints a `https://...trycloudflare.com` URL and a
**generated** token (paste both into the laptop command; the same token must be
typed nowhere else). Before the tunnel it also answers one Banglish question
through the endpoint, so what the desktop app will show is on the screen in this
notebook, from the same code path as section 5's 40.

```bash
# Linux and macOS
cd biai_bank_assistant
python3 -m pip install -r requirements.txt
BIAI_API_URL="https://<printed-url>" BIAI_API_TOKEN="<printed-token>" python3 main.py
```

```powershell
# Windows PowerShell
cd biai_bank_assistant
py -3 -m pip install -r requirements.txt
$env:BIAI_API_URL   = "https://<printed-url>"
$env:BIAI_API_TOKEN = "<printed-token>"
py -3 main.py
```

The pill then reads **MODEL READY — KAGGLE** and the inspector shows the
endpoint's retrieval mode and generator, which is what makes the desktop replies
the ones in `reports/offline_results.json` rather than the laptop's own English
FAQ text. `BIAI_ADAPTER_PATH` is only needed if you also ran section 6 and want
the server to use that adapter. If the tunnel dies mid-conversation the window
says so and keeps answering from the FAQ on the laptop, so the demo never becomes
a blank box. Kaggle gives no inbound port; that is why there is a tunnel, and why
`api_server.py` refuses requests without the token.

Getting the two values, if you have never done this: after the cell finishes, the
last lines are `BIAI_API_URL=https://…trycloudflare.com` and
`BIAI_API_TOKEN=…`. Copy them as printed — the token is generated for this
session by the cell itself, is never shown again, and is not a token from any
token-generator website, because `api_server.py` compares the `Authorization`
header to the value this cell put in its own environment. Wrong token is a `401`
in the laptop terminal; a stopped notebook is a `LOCAL FAQ` pill.

The tunnel is a demo path, not hosting: it lives only while this session runs, and
Kaggle recycles idle sessions. A packaged desktop app cannot be a public web demo
by itself — the public link to send is the Loom recording, with this section shown
inside it.


In [ ]:
# Serves api_server.py over the staged project and publishes it with a Cloudflare
# quick tunnel (no account, no domain), and keeps running so the desktop app can
# talk to this notebook. Re-runnable: the previous server and tunnel started by
# this cell are stopped first, uvicorn is polled on its own port before any
# tunnel is opened, cloudflared is read from a log file instead of a blocking
# pipe, and the edge is forced to IPv4. Those four things are what produced the
# error spam in earlier runs: a second uvicorn on a busy port, a blocking
# readline that looked like a hang, and a healthy tunnel that Kaggle itself could
# not re-open over IPv6.
import json
import secrets
import socket
import subprocess
import sys
import time
import urllib.request

os.chdir(DST)
API_PORT = int(os.environ.get('BIAI_API_PORT', '8000'))
API_TOKEN = os.environ.get('BIAI_API_TOKEN') or secrets.token_urlsafe(18)
os.environ['BIAI_API_TOKEN'] = API_TOKEN
!pip -q install fastapi uvicorn


def _stop(proc):
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()


for _name in ('_nb_server', '_nb_tunnel'):
    _stop(globals().pop(_name, None))

if not (DST / 'api_server.py').exists():
    print('api_server.py is not in the staged project: re-upload the dataset with it.')
elif socket.socket().connect_ex(('127.0.0.1', API_PORT)) == 0:
    print(f'port {API_PORT} is already taken by something this notebook does not own; '
          f'set BIAI_API_PORT to another port and re-run.')
else:
    server_log = open(DST / 'api_server.log', 'w')
    _nb_server = subprocess.Popen(
        [sys.executable, '-m', 'uvicorn', 'api_server:app', '--host', '127.0.0.1',
         '--port', str(API_PORT), '--log-level', 'warning'],
        cwd=str(DST), stdout=server_log, stderr=subprocess.STDOUT, text=True)

    # Qwen and the translator load while api_server.py is imported, which costs a
    # minute on a T4. Waiting on the local port means the tunnel is never opened
    # against a server that is not listening yet.
    health, local = None, f'http://127.0.0.1:{API_PORT}'
    for _ in range(180):
        time.sleep(2)
        if _nb_server.poll() is not None:
            print('uvicorn exited early; last lines of its log:')
            print(''.join(open(DST / 'api_server.log', errors='ignore').readlines()[-25:]))
            break
        try:
            with urllib.request.urlopen(f'{local}/health', timeout=5) as response:
                health = response.read().decode()
                break
        except Exception:
            continue
    if health:
        print('endpoint live in the notebook:', health)
        # One Banglish question, answered here, proves the served path is the same
        # one section 5 measured: dense retrieval, Qwen, and the renderer.
        req = urllib.request.Request(
            f'{local}/answer', data=json.dumps({'question': 'Card er PIN bhule gechi, reset korbo kibhabe?'}).encode(),
            headers={'Content-Type': 'application/json', 'Authorization': f'Bearer {API_TOKEN}'},
            method='POST')
        try:
            with urllib.request.urlopen(req, timeout=180) as response:
                print('warm-up answer:', json.loads(response.read())['answer'])
        except Exception as exc:
            print('warm-up request failed:', type(exc).__name__, exc)

        if not (DST / 'cloudflared').exists():
            !curl -sL -o cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
            !chmod +x cloudflared
        tunnel_log = open(DST / 'tunnel.log', 'w')
        _nb_tunnel = subprocess.Popen(
            ['./cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{API_PORT}',
             '--no-autoupdate', '--edge-ip-version', '4'],
            cwd=str(DST), stdout=tunnel_log, stderr=subprocess.STDOUT, text=True)
        url = None
        for _ in range(60):
            time.sleep(2)
            text = open(DST / 'tunnel.log', errors='ignore').read()
            if text.find('https://') >= 0:
                url = text[text.index('https://'):].split()[0].rstrip('|,')
                break
            if _nb_tunnel.poll() is not None:
                print('cloudflared exited; its log says:')
                print(text[-1500:])
                break
        if not url:
            print('no tunnel URL: Kaggle blocked the cloudflared download or the handshake')
        else:
            try:
                with urllib.request.urlopen(f'{url}/health', timeout=30) as response:
                    print('public health:', response.read().decode())
            except Exception as exc:
                print('the URL is published; /health had not answered yet:', type(exc).__name__)
            print()
            print('On the laptop, run (python3 on Linux/macOS, py -3 on Windows):')
            print(f'  BIAI_API_URL={url} BIAI_API_TOKEN={API_TOKEN} python3 main.py')
            print(f'(server pid {_nb_server.pid}, tunnel pid {_nb_tunnel.pid}; '
                  f're-running this cell, or stopping the session, ends both)')


## 9. Submission checklist

| Deliverable | Where it lives | Weight (brief's own table) |
|---|---|---|
| Loom video, ≤5 minutes | record it from section 4 and section 5 | 20% |
| Works on the supplied sample files | `reports/offline_results.json` | 25% |
| Clear build that handles mistakes | `src/assistant.py` gate order + the hand-off reasons printed above | 20% |
| Hardest part solved | cross-script retrieval + the groundedness guards | 15% |
| Clean code runnable from the README | `README.md` | 10% |
| Honesty in the note | `ONE_PAGE_NOTE.md`, "What does not work well" | 10% |

After the run: paste section 5's numbers into the **Model stack** column of `ONE_PAGE_NOTE.md` and section 7's table into its comparison paragraph. Both are still `pending run` until you do.

Before pasting anything: section 5's **per script** table has to show Banglish answered. The
overall gate accuracy is the number that hid a run where no Banglish question was answered at
all, so a healthy aggregate with an empty Banglish row is a failure, not a result.
